In [1]:
import pandas as pd
import requests

def get_russian_inflation():
    # Открытый API СтатБюро с данными Росстата
    url = "https://www.statbureau.org/get-data-json?country=russia"
    
    print("Скачиваем исторические данные об инфляции...")
    response = requests.get(url)
    
    # Проверяем успешность запроса
    if response.status_code != 200:
        raise Exception(f"Ошибка при получении данных. Код: {response.status_code}")
        
    # Преобразуем полученный JSON в список словарей
    data = response.json()
    
    # Создаем DataFrame (таблицу) из JSON
    df = pd.DataFrame(data)
    
    # В ответе API много полей, нас интересует отформатированная дата и сама ставка за месяц
    # MonthFormatted: дата в формате YYYY-MM-DD
    # InflationRate: уровень инфляции за этот месяц (в процентах)
    df = df[['MonthFormatted', 'InflationRate']]
    
    # Переименовываем колонки для удобства
    df.columns = ['Дата', 'Инфляция_за_месяц_%']
    
    # Преобразуем колонку "Дата" в формат datetime (всегда первое число месяца)
    df['Дата'] = pd.to_datetime(df['Дата'])
    
    # Сортируем данные хронологически (от 1991 года до наших дней)
    df = df.sort_values(by='Дата').reset_index(drop=True)
    
    return df

def calculate_annual_inflation(df):
    """
    Дополнительная функция, если вам нужна годовая инфляция (г/г).
    Считает сумму инфляции за последние 12 месяцев.
    """
    # Создаем копию, чтобы не менять исходный датафрейм
    annual_df = df.copy()
    
    # Расчет годовой инфляции сложным процентом: (1 + i_1)*(1 + i_2)*...*(1 + i_12) - 1
    # Для этого переведем проценты в доли, прибавим 1, найдем произведение за 12 месяцев
    annual_df['Множитель'] = 1 + (annual_df['Инфляция_за_месяц_%'] / 100)
    annual_df['Годовая_инфляция_%'] = (annual_df['Множитель'].rolling(window=12).apply(lambda x: x.prod(), raw=True) - 1) * 100
    
    # Округляем до двух знаков
    annual_df['Годовая_инфляция_%'] = annual_df['Годовая_инфляция_%'].round(2)
    
    # Убираем вспомогательную колонку
    annual_df = annual_df.drop(columns=['Множитель'])
    
    return annual_df

if __name__ == "__main__":
    try:
        # Получаем данные по месячной инфляции
        inflation_df = get_russian_inflation()
        
        # Добавляем расчет годовой инфляции (к соответствующему месяцу прошлого года)
        inflation_df = calculate_annual_inflation(inflation_df)
        
        print("\nПоследние данные по инфляции:")
        print(inflation_df.tail(10))
        
        # Сохраняем в CSV
        filename = "russia_inflation_history.csv"
        inflation_df.to_csv(filename, index=False, encoding='utf-8')
        print(f"\nДанные успешно сохранены в файл {filename}")
        
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Скачиваем исторические данные об инфляции...

Последние данные по инфляции:
          Дата  Инфляция_за_месяц_%  Годовая_инфляция_%
412 2025-05-01                 0.43                9.89
413 2025-06-01                 0.20                9.41
414 2025-07-01                 0.57                8.80
415 2025-08-01                -0.40                8.14
416 2025-09-01                 0.34                7.99
417 2025-10-01                 0.50                7.73
418 2025-11-01                 0.42                6.65
419 2025-12-01                 0.32                5.60
420 2026-01-01                 1.62                6.01
421 2026-02-01                 0.73                5.92

Данные успешно сохранены в файл russia_inflation_history.csv


In [2]:
import pandas as pd
import requests

def get_weekly_inflation():
    # Ссылка на страницу ЦБ с оценками инфляции на основе данных Росстата
    # Эта страница содержит таблицу с недельным изменением цен (в %)
    url = "https://cbr.ru/hd_base/infl/weekly_inflation/"
    
    # Заголовки, чтобы сайт не блокировал скрипт
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print("Загружаем данные по недельной инфляции...")
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"Не удалось получить данные. Код ошибки: {response.status_code}")

    # Читаем таблицы с веб-страницы
    tables = pd.read_html(response.text)
    
    # Нужная таблица обычно первая на странице
    df = tables[0]

    # Приводим названия колонок в порядок
    # Обычно там: 0 - Период (неделя), 1 - Инфляция за неделю (%)
    df.columns = ['Период', 'Инфляция_недельная_%']

    # Очистка данных
    # Заменяем запятые на точки для корректной конвертации в числа
    df['Инфляция_недельная_%'] = df['Инфляция_недельная_%'].astype(str).str.replace(',', '.').astype(float)
    
    # Для удобства разделим колонку 'Период' на начало и конец недели (если нужно)
    # Формат в таблице обычно: "дд.мм.гггг - дд.мм.гггг"
    df[['Начало_недели', 'Конец_недели']] = df['Период'].str.split(' - ', expand=True)
    
    # Превращаем в формат даты
    df['Начало_недели'] = pd.to_datetime(df['Начало_недели'], dayfirst=True)
    df['Конец_недели'] = pd.to_datetime(df['Конец_недели'], dayfirst=True)

    # Сортируем от новых к старым (или наоборот)
    df = df.sort_values(by='Конец_недели', ascending=False).reset_index(drop=True)

    return df

if __name__ == "__main__":
    try:
        data = get_weekly_inflation()
        
        print("\nПоследние 5 недель:")
        print(data[['Период', 'Инфляция_недельная_%']].head())
        
        # Сохранение
        data.to_csv("weekly_inflation_russia.csv", index=False, encoding="utf-8-sig")
        print("\nФайл 'weekly_inflation_russia.csv' успешно сохранен.")
        
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Загружаем данные по недельной инфляции...
Произошла ошибка: Не удалось получить данные. Код ошибки: 404
